# 11 — TCAV (Concept-Based Interpretability)

**Inputs needed**
- `models/saved/best_swinvit_model.pth` (SwinViT trained in Phase 4).
- `data/raw_radiomics_features.csv` and `data/varfs_selected_features.json` (Phase 3).
- `data/processed/<PID>/before_cropped.nii.gz` for every labelled patient.

**Outputs produced**
- `tcav/cavs.npz` and `tcav/train_accuracies.json`.
- `results/tcav_per_feature_scores.json` and `results/tcav_per_family_scores.json`.
- `Plots/tcav_per_feature_bar.png`, `Plots/tcav_per_family_bar.png`, `Plots/tcav_concept_heatmap.png`.

**What it does**
1. Builds two concept families from the radiomics tables: per-feature (top-N stable VaRFS features) and per-family (PCA-1 of every PyRadiomics class).
2. Hooks SwinViT's deepest stage to extract per-volume activations.
3. Trains a linear CAV per concept (`LogisticRegression`).
4. Computes TCAV scores via directional derivatives and a permutation p-value against random concepts.
5. Plots and saves the results.

**Expected runtime:** ~5–15 minutes for ~50 patients on GPU; ~2× longer on CPU.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.config import ensure_dirs, load_config, set_seed
from src.utils.logger import log_dict, log_section, setup_logger, get_logger

cfg = load_config(ROOT / "configs" / "default.yaml")
ensure_dirs(cfg)
set_seed(int(cfg.get("seed", 42)))
setup_logger("hcc", log_file=Path(cfg["paths"]["logs_dir"]) / "tcav.log", console_level="INFO")
logger = get_logger("hcc.tcav.notebook")
log_dict(logger, "config.tcav", cfg.get("tcav", {}))

## 1. Load SwinViT checkpoint

The TCAV core hooks one of SwinViT's submodules. The default target is `encoder.layers3` (deepest Swin stage). If the resolved name differs, the helper logs a warning and falls back to the deepest matching layer.

In [ ]:
import torch
from src.models.swin_vit import SwinViT3D
from src.utils.tcav import TCAV3D, RadiomicsConceptBuilder

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt = ROOT / cfg["paths"]["model_save_dir"] / "best_swinvit_model.pth"
model = SwinViT3D(
    img_size=tuple(cfg["swin_vit"]["img_size"]),
    patch_size=tuple(cfg["swin_vit"]["patch_size"]),
    in_channels=int(cfg["swin_vit"]["in_channels"]),
    embed_dim=int(cfg["swin_vit"]["embed_dim"]),
    depths=tuple(cfg["swin_vit"]["depths"]),
    num_heads=tuple(cfg["swin_vit"]["num_heads"]),
    dropout=float(cfg["swin_vit"]["dropout"]),
).to(device)
bundle = torch.load(ckpt, map_location=device)
model.load_state_dict(bundle.get("model_state", bundle))
model.eval()

tcav = TCAV3D(
    model=model,
    target_layer_name=cfg["tcav"]["target_layer"],
    target_class=int(cfg["tcav"]["target_class"]),
    device=device,
)
logger.info("Hooked layer: %s", tcav.target_layer_name)

## 2. Extract activations for every labelled patient

In [ ]:
from src.data.dataset import HCCDataset

dataset = HCCDataset(
    processed_dir=cfg["paths"]["processed_dir"],
    labels_csv=cfg["paths"]["labels_csv"],
    target_size=tuple(cfg["swin_vit"]["img_size"]),
)
activations, pids = tcav.extract_activations(dataset)
pid_to_index = {pid: i for i, pid in enumerate(pids)}
logger.info("activations shape=%s, n=%d", activations.shape, len(pids))

## 3. Build concepts (per-feature, per-family, random)

In [ ]:
builder = RadiomicsConceptBuilder(quartile=float(cfg["tcav"]["quartile_split"]))
radiomics_csv = ROOT / "data" / "raw_radiomics_features.csv"
selected_json = ROOT / "data" / "varfs_selected_features.json"
labels_csv = ROOT / cfg["paths"]["labels_csv"]

per_feature = builder.build_per_feature_concepts(
    radiomics_csv=radiomics_csv,
    selected_features_json=selected_json,
    top_n=int(cfg["tcav"]["top_n_features"]),
)
per_family, signs = builder.build_per_family_concepts(
    radiomics_csv=radiomics_csv,
    labels_csv=labels_csv,
    families=list(cfg["tcav"]["feature_families"]),
)
random_concepts = builder.build_random_concepts(
    all_pids=pids,
    n_concepts=int(cfg["tcav"]["n_random_concepts"]),
    size=int(cfg["tcav"]["random_concept_size"]),
    seed=int(cfg.get("seed", 42)),
)
logger.info(
    "per_feature=%d, per_family=%d, random=%d",
    len(per_feature), len(per_family), len(random_concepts),
)

## 4. Train CAVs and verify they are non-trivial

In [ ]:
cavs_feat = tcav.build_cavs(activations, pid_to_index, per_feature)
cavs_fam = tcav.build_cavs(activations, pid_to_index, per_family)
cavs_random = tcav.build_cavs(activations, pid_to_index, random_concepts)

import pandas as pd
rows = []
for kind, store in [("feature", cavs_feat), ("family", cavs_fam)]:
    for name, rec in store.items():
        rows.append({
            "kind": kind,
            "concept": name,
            "train_accuracy": rec.train_accuracy,
            "n_pos": rec.n_positive,
            "n_neg": rec.n_negative,
        })
pd.DataFrame(rows).sort_values("train_accuracy", ascending=False).head(15)

## 5. Compute TCAV scores + permutation p-values

In [ ]:
max_samples = cfg["tcav"].get("max_samples")
max_samples = None if max_samples in (None, "null") else int(max_samples)
random_scores = [
    tcav.compute_tcav_score(dataset, rec.vector, max_samples=max_samples)
    for rec in cavs_random.values()
]

def _score_concepts(cavs):
    out = {}
    for name, rec in cavs.items():
        score = tcav.compute_tcav_score(dataset, rec.vector, max_samples=max_samples)
        p = TCAV3D.compute_significance(score, random_scores)
        out[name] = {"score": score, "p_value": p, "train_accuracy": rec.train_accuracy}
    return out

scores_feat = _score_concepts(cavs_feat)
scores_fam = _score_concepts(cavs_fam)
log_section(logger, "Per-feature scores")
log_dict(logger, "per_feature", scores_feat)
log_section(logger, "Per-family scores")
log_dict(logger, "per_family", scores_fam)

## 6. Persist + plot

In [ ]:
import json
from src.utils.tcav import save_cav_accuracies, save_cavs_npz
from src.utils.visualization import plot_tcav_scores, plot_concept_importance_heatmap

tcav_dir = ROOT / cfg["paths"]["tcav_dir"]
results_dir = ROOT / cfg["paths"]["results_dir"]
plots_dir = ROOT / cfg["paths"]["plots_dir"]

all_cavs = {**cavs_feat, **cavs_fam}
save_cavs_npz(all_cavs, tcav_dir / "cavs.npz")
save_cav_accuracies(all_cavs, tcav_dir / "train_accuracies.json")

with open(results_dir / "tcav_per_feature_scores.json", "w") as fh:
    json.dump(scores_feat, fh, indent=2)
with open(results_dir / "tcav_per_family_scores.json", "w") as fh:
    json.dump(scores_fam, fh, indent=2)

plot_tcav_scores(
    {k: v["score"] for k, v in scores_feat.items()},
    {k: v["p_value"] for k, v in scores_feat.items()},
    plots_dir / "tcav_per_feature_bar.png",
    title="TCAV (per-feature)",
)
plot_tcav_scores(
    {k: v["score"] for k, v in scores_fam.items()},
    {k: v["p_value"] for k, v in scores_fam.items()},
    plots_dir / "tcav_per_family_bar.png",
    title="TCAV (per-family)",
)
plot_concept_importance_heatmap(
    {k: v["score"] for k, v in scores_feat.items()},
    {k: v["score"] for k, v in scores_fam.items()},
    plots_dir / "tcav_concept_heatmap.png",
)

## 7. Sanity check — gradients flow through the hooked layer

If this prints `True` we have non-zero gradients at the target layer, which is required for directional derivatives. If it prints `False`, double-check the `tcav.target_layer` config: it must be on the forward path of the model.

In [ ]:
import numpy as np
sample = dataset[0]["image"].unsqueeze(0)
fake_cav = np.random.RandomState(0).randn(activations.shape[1]).astype(np.float32)
fake_cav /= np.linalg.norm(fake_cav)
deriv = tcav.directional_derivative(sample, fake_cav)
print("Directional derivative non-zero:", abs(deriv) > 0)